## Intermediate Data Science

## Important Information

- Email: [joanna_bieri@redlands.edu](mailto:joanna_bieri@redlands.edu)
- Office Hours take place in Duke 209 - [Office Hours Schedule](https://joannabieri.com/schedule.html)
- [Class Website](https://joannabieri.com/data201_intermediate.html)
- [Syllabus](https://joannabieri.com/data201/IntermediateDataScience.pdf)

## Today's Reading

*Introduction to Machine Learning with Python*, Chapter 2.3.7 - Kernelized Support Vector Machines.

## Career Reading

*Build a Career in Data Science*, 2.1 to 2.5 Types of Data Science Companies. Read it and take notes. We discuss all five companies together in class today, so bring your notes!

## Final Project

Your **Project Proposal and Exploratory Data Analysis** is due **tonight, Tuesday 11/10 at 11:59pm**. It is individual work, on Canvas and in your sandbox. See the [Final Project Information](https://joannabieri.com/data201/FinalProjectInfo.html) page.

## Video

[Video - Support Vector Machines](https://youtu.be/PsCjpB8ROkY)

If you want to see the ideas explained another way, this short video (not mine) is really good:

- [StatQuest - Support Vector Machines Part 1: Main Ideas](https://www.youtube.com/watch?v=efR1C6CvhmE)

---------------------------------------

## Continue with Supervised Learning

So far we have KNN, linear and logistic regression, Naive Bayes, and decision trees. Today we add one more classifier: **Support Vector Machines (SVMs)**. An SVM draws the boundary between two classes that is as far away from both classes as possible. With a trick (the kernel trick) it can draw curved boundaries too. At the end we will try it on a bigger example, handwritten digits, and compare it to a decision tree.

You will see a lot of the same ideas from the last few classes: a parameter that controls how complicated the model is, overfitting if we push that parameter too far, and a validation set to help us choose the best value.

In [ ]:
# NOTE - This list of package imports is getting long
# In a professional setting you would only want to 
#      import what you need!
# I had chatGPT break the packages into groups here

# ============================================================
# Basic packages
# ============================================================
import os                             # For file and directory operations
import numpy as np                    # For numerical computing and arrays
import pandas as pd                   # For data manipulation and analysis

# ============================================================
# Visualization packages
# ============================================================
import matplotlib.pyplot as plt        # Static 2D plotting
from matplotlib.colors import ListedColormap
import seaborn as sns                  # Statistical data visualization built on matplotlib

# Interactive visualization with Plotly
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'notebook_connected'   # Set renderer for interactive output in notebooks

# ============================================================
# Scikit-learn: Core utilities for model building and evaluation
# ============================================================
from sklearn.model_selection import train_test_split    # Train/test data splitting
from sklearn.preprocessing import PolynomialFeatures, MinMaxScaler, StandardScaler  # Feature transformations and scaling
from sklearn.metrics import (                            # Model evaluation metrics
    mean_squared_error, r2_score, accuracy_score, 
    precision_score, recall_score, confusion_matrix, 
    classification_report
)

# ============================================================
# Scikit-learn: Linear and polynomial models
# ============================================================
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor       # For KNN

# ============================================================
# Scikit-learn: Synthetic dataset generators
# ============================================================
from sklearn.datasets import make_classification, make_regression, make_blobs, make_circles
from sklearn.datasets import load_digits

# ============================================================
# Scikit-learn: Naive Bayes models
# ============================================================
from sklearn.naive_bayes import GaussianNB, BernoulliNB, MultinomialNB

# ============================================================
# Scikit-learn: Decision Trees
# ============================================================
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree

# ============================================================
# Scikit-learn: Support Vector Machines - NEW today
# ============================================================
from sklearn.svm import SVC

Here is a helper function that draws the decision boundary of any classifier on 2D data. It colors the whole plane by what the model would predict there. For an SVM it also draws the margin (dashed lines) and circles the support vectors.

In [ ]:
def plot_boundary(model, X, y, title="", margin=False):
    # make a grid of points that covers the data
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                         np.linspace(y_min, y_max, 300))
    grid = np.c_[xx.ravel(), yy.ravel()]

    # color the plane by the prediction at every grid point
    Z = model.predict(grid).reshape(xx.shape)
    plt.contourf(xx, yy, Z, alpha=0.25, cmap='coolwarm')
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', edgecolors='k')

    if margin:
        # the boundary is where decision_function = 0, the margin is where it is -1 and +1
        D = model.decision_function(grid).reshape(xx.shape)
        plt.contour(xx, yy, D, levels=[-1, 0, 1], colors='k', linestyles=['--', '-', '--'])
        # circle the support vectors
        sv = model.support_vectors_
        plt.scatter(sv[:, 0], sv[:, 1], s=200, facecolors='none', edgecolors='k', linewidths=1.5)

    plt.title(title)
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.show()

---------------------------------------

## Support Vector Machines

Today we are going to talk about Support Vector Machines (SVMs) for classification. SVMs work really well on small and medium sized data sets, and with the kernel trick they can draw curved boundaries.

### The Core Idea: A Margin-Based Classifier

Imagine you have data points that belong to two classes. There are lots of lines that would separate them, so which one should we pick? An SVM chooses the line that gives the **largest margin**.

The **margin** is the distance between the boundary and the nearest data points from each class. These nearest points are called **support vectors**. They are the only points that matter to the boundary! If you moved any of the other points around (without crossing the margin) the boundary would not change at all.

A larger margin usually means the model will generalize better to new data, because new points have more room before they land on the wrong side.

Let's make some data with two classes.

In [ ]:
X, y = make_blobs(n_samples=60, centers=2, cluster_std=1.2, random_state=6)

plt.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', edgecolors='k')
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Two Classes")
plt.show()

Fitting an SVM looks just like every other model we have used in sklearn: first we make the model, then we call `.fit()` sending in the training data..

NEW COMMAND? `SVC` stands for Support Vector Classifier. `kernel='linear'` tells it we want a straight line boundary between the classes. $C$ is a paramter that tells the SVM how "expensive" a mistake is, we will talk more about this and how the model is fit below. 

In [ ]:
svm = SVC(kernel='linear', C=1)   # a straight line boundary, C = 1
svm.fit(X, y)                     # find the line with the biggest margin

# support_vectors_ holds the points that decide the boundary
print("Number of support vectors:", len(svm.support_vectors_))
plot_boundary(svm, X, y, title="Linear SVM, C = 1", margin=True)

The solid line is the boundary, the dashed lines are the edges of the margin, and the circled points are the support vectors. Only 3 of the 60 points decide where this line goes!

So how does the computer find this line? Here is the math.

### Show me the math! 

For a binary classification problem the labels are $y \in \{-1, +1\}$, the point belongs to one class or the other, and we want a hyperplane (a line in 2D) that marks the zero (aka. along the line y=0):

$$
w \cdot x + b = 0
$$

where $w$ is the weight vector (it sets the direction of the line) and $b$ is the bias (it shifts the line). With two features this is just

$$
w_1 x_1 + w_2 x_2 + b = 0
$$

which is the equation of a plane, the same as what we had in logistic regression on Day 12!

NOTE: sklearn calls our two classes 0 and 1. In the math we call them $-1$ and $+1$, because it makes the formulas below much simpler. Class 0 is $-1$ and class 1 is $+1$.

The classifier predicts

$$
\hat{y} = \text{sign}(w \cdot x + b)
$$

The **sign function** just tells you if a number is positive or negative:

$$
\text{sign}(z) = \begin{cases} +1 & \text{if } z > 0 \\ -1 & \text{if } z < 0 \end{cases}
$$

eg. $\text{sign}(3.7) = +1\;\;$ and $\;\;\text{sign}(-0.2) = -1\;\;$. So you plug the point into $w \cdot x + b$, and you get $+1$ on one side of the line and $-1$ on the other.

Let's look at the $w$ and $b$ that our SVM found above. sklearn saves them in `coef_` and `intercept_`.

In [ ]:
# w and b from the SVM we just trained (C = 1)
w = svm.coef_[0]        # the weight vector, one number for each feature
b = svm.intercept_[0]   # the bias, one number
print("w =", w)
print("b =", b)

Now let's draw it using matplotlib. The boundary is where $w \cdot x + b = 0$ and the edges of the margin are where $w \cdot x + b = -1$ and $w \cdot x + b = +1$. The arrow is the vector $w$. Notice that it points straight across the boundary, toward the $+1$ class. This is the same data and the same model as above, and the circled points are the same 3 support vectors. They sit exactly on the dashed lines.

In [ ]:
### JUST RUN THIS CODE - DEMO ONLY
# x1 values that go across the plot
x1 = np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 100)

plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', edgecolors='k')

# solve w1*x1 + w2*x2 + b = level for x2, for each of the three lines
for level, style in [(-1, '--'), (0, '-'), (1, '--')]:
    x2 = (level - b - w[0] * x1) / w[1]
    plt.plot(x1, x2, 'k' + style)
    plt.text(x1[-1] + 0.1, x2[-1], f"$w_1x_1 + w_2x_2 + b = {level}$", va='center')

# draw the vector w, starting from a point on the boundary
x_start = X[:, 0].mean()
y_start = (-b - w[0] * x_start) / w[1]
# (w is drawn 2 times longer so it is easier to see)
plt.arrow(x_start, y_start, 2 * w[0], 2 * w[1], width=0.08, head_width=0.4, head_length=0.4,
          color='green', length_includes_head=True)
plt.text(x_start - 0.7, y_start + w[1], "w", color='green', fontsize=18)

# circle the support vectors
sv = svm.support_vectors_
plt.scatter(sv[:, 0], sv[:, 1], s=200, facecolors='none', edgecolors='k', linewidths=1.5)

# label the two sides
plt.text(X[:, 0].mean(), X[:, 1].max() + 1, "$w_1x_1 + w_2x_2 + b < 0$, predict -1 (class 0)", ha='center')
plt.text(X[:, 0].mean(), X[:, 1].min() - 1.5, "$w_1x_1 + w_2x_2 + b > 0$, predict +1 (class 1)", ha='center')

plt.xlim(X[:, 0].min() - 1, X[:, 0].max() + 6)   # room for the labels
plt.ylim(X[:, 1].min() - 2, X[:, 1].max() + 2)
plt.xlabel("$x_1$ (Feature 1)")
plt.ylabel("$x_2$ (Feature 2)")
plt.title("The boundary, the margin, and w")
plt.show()

For correctly classified points that are outside the margin we ask for

$$
y_i (w \cdot x_i + b) \ge 1
$$

Why do we want this to be greater than or equal to 1? Let's break it into two pieces.

1. **Is the point on the correct side?** Multiplying by the label $y_i$ checks this. If the point is in class $+1$ we need $w \cdot x_i + b$ to be positive, and if it is in class $-1$ we need $w \cdot x_i + b$ to be negative. Either way, the product $y_i (w \cdot x_i + b)$ is positive when the point is on the correct side and negative when it is on the wrong side. One formula covers both classes, that is why we use $-1$ and $+1$!
2. **Is it far enough from the line?** If we only asked for the product to be bigger than 0, a point sitting right next to the boundary would count as fine. Asking for 1 or more means the point has to be on the correct side AND past the dashed line for its class. So the dashed lines are exactly where $w \cdot x + b = -1$ and $+1$.

Why 1 and not some other number? You can multiply $w$ and $b$ by any number (try 2) and you get the exact same line. So we pick the size of $w$ that makes the closest points land at exactly $+1$ and $-1$. That choice is what makes the math below work.

Here are some made up examples:

| label $y_i$ | $w \cdot x_i + b$ | $y_i (w \cdot x_i + b)$ | What it means |
|---|---|---|---|
| $+1$ | $2.5$ | $2.5$ | correct side, outside the margin, GOOD |
| $-1$ | $-1$ | $1$ | correct side, right on the dashed line (a support vector!) |
| $-1$ | $-0.4$ | $0.4$ | correct side, but INSIDE the margin |
| $+1$ | $-0.3$ | $-0.3$ | WRONG side of the line |

How wide is the margin? Because we picked the size of $w$ so the dashed lines are at $+1$ and $-1$, the width of the margin works out to be

$$
\frac{2}{\|w\|}
$$

where $\|w\|$ is the length of the vector $w$. With two features this is just the Pythagorean theorem:

$$
\|w\| = \sqrt{w_1^2 + w_2^2}
$$

(If you have had Calculus III or Linear Algebra, the $\frac{2}{\|w\|}$ comes from the formula for the distance from a point to a line.) The important thing to notice: a SMALLER $w$ gives a WIDER margin.

To make the margin as BIG as possible, we make $\|w\|$ as small as possible:

$$
\begin{aligned}
\text{Minimize:} \quad & \frac{1}{2} \|w\|^2 \\
\text{Subject to:} \quad & y_i (w \cdot x_i + b) \ge 1, \quad \text{for every } i
\end{aligned}
$$

This is a **convex optimization problem**. It has one global minimum (think of a bowl shaped function from calculus), so the computer can always find the best line.

### How does the computer pick the support vectors?

We never tell the computer which points are the support vectors. It solves the minimization problem above for $w$ and $b$, and the support vectors are the points that end up exactly on the dashed lines, where $y_i (w \cdot x_i + b) = 1$. These are the points that are "holding up" the margin. If you made the margin any wider, they would be inside it.

When the computer solves this problem it actually writes $w$ as a combination of the training points:

$$
w = \sum_i \alpha_i y_i x_i
$$

and it finds a number $\alpha_i \ge 0$ for every point. For almost every point $\alpha_i = 0$, so that point does not show up in $w$ at all. The points with $\alpha_i > 0$ are the support vectors! In our example above only three $\alpha_i$ values were non-zero and we had three support vectors.


With a soft margin (next section) the support vectors are the points on the dashed lines AND any points inside the margin or on the wrong side. So when the margin gets wider, more points become support vectors.

### Soft Margins and the C parameter

Real data is not always perfectly separable. To allow some points to be inside the margin (or even on the wrong side) we add **slack variables** $\xi_i$ and a penalty $C$:

$$
\text{Minimize: } \frac{1}{2} \|w\|^2 + C \sum_i \xi_i
\qquad \text{Subject to: } y_i (w \cdot x_i + b) \ge 1 - \xi_i, \quad \xi_i \ge 0
$$

$C$ controls the trade-off between a wide margin and fewer mistakes on the training data. Look at the formula: $C$ multiplies the slack, so it sets how big the penalty is for each point inside the margin.

- **Large C**: a big penalty for every mistake, so you get a narrow margin and fewer training errors (risk of overfitting)
- **Small C**: a small penalty for every mistake, so you get a wider margin and more training errors (risk of underfitting)

Let's try a small C on the same data.

NOTE: This is the same idea as alpha in Ridge and Lasso (Day 13), but backwards! A BIG alpha meant a simpler model. A SMALL C means a simpler model.

In [ ]:
svm = SVC(kernel='linear', C=0.01)   # same data, much smaller C
svm.fit(X, y)

print("Number of support vectors:", len(svm.support_vectors_))
plot_boundary(svm, X, y, title="Linear SVM, C = 0.01", margin=True)

With $C = 0.01$ the margin got much wider and now 22 points are support vectors. Lots of points are allowed inside the margin. The model still gets every training point right, but the boundary moved.

## You Try

Change `C` in the cell above to 0.1, 1, and 100. Watch the number of support vectors and the width of the margin. What happens between $C=1$ and $C=100$? Why do you think that is?

---------------------------------------

## The Kernel Trick - Curved Boundaries

So far our classes could be split with a straight line. What if the data can not be separated by a straight line at all? Here is data where one class makes a ring around the other.

In [ ]:
X, y = make_circles(n_samples=400, factor=0.4, noise=0.2, random_state=42)

plt.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', edgecolors='k')
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Circle Data")
plt.show()

We are going to choose a parameter in this section, so let's split the data right away, just like we did on Day 13. We make a **Training Set**, a **Validation Set**, and a **Testing Set** (80% / 10% / 10%). We use the validation set to choose the parameter, and we only get to use the test set ONE time, at the very end.

In [ ]:
random_seed = 42
# First split: hold out test data
X_train, X_test_full, y_train, y_test_full = train_test_split(
    X, y, test_size=0.2, random_state=random_seed
)

# Second split: create a validation set from the testing data
X_test, X_val, y_test, y_val = train_test_split(
    X_test_full, y_test_full, test_size=0.5, random_state=random_seed
)
# (Hold out data so that final proportions are 80% train / 10% val / 10% test)

print(f"Train: {len(X_train)}, Validation: {len(X_val)}, Test: {len(X_test)}")

In [ ]:
# first try a straight line
svm = SVC(kernel='linear')
svm.fit(X_train, y_train)

# score on the VALIDATION data, not the test data
print("Accuracy on the validation data:", accuracy_score(y_val, svm.predict(X_val)))
plot_boundary(svm, X_train, y_train, title="Linear SVM on circle data")

A straight line does no better than flipping a coin (0.5)! There is no straight line that separates a ring from the points in the middle of it.

### The trick

Here is the idea. What if we add a new feature? For the ring data let's make a third feature

$$
x_3 = x_1^2 + x_2^2
$$

This is the distance from the center, squared. The points in the middle are close to the center, so they get small values. The points in the ring are far from the center, so they get big values. Let's plot all three features and see what happens!

In [ ]:
### JUST RUN THIS CODE - DEMO ONLY
# make the new feature: distance from the center, squared
x3 = X_train[:, 0]**2 + X_train[:, 1]**2

# put everything in a DataFrame so we can plot it in 3D
df_3d = pd.DataFrame({'x1': X_train[:, 0], 'x2': X_train[:, 1], 'x3': x3,
                      'class': y_train.astype(str)})
fig = px.scatter_3d(df_3d, x='x1', y='x2', z='x3', color='class', height=600,
                    title="Circle data with a new feature x3")
fig.update_traces(marker=dict(size=3))
fig.show()

Spin the plot around! The middle class sits at the bottom and the ring class is lifted up. Now a flat sheet (a plane) can go between them. A flat boundary in 3D turns into a circle when you look back down at the original 2 features.

This is what a **kernel** does for an SVM. It lets the SVM work as if we had added new features like $x_3$. The **kernel trick** is that the computer never has to actually make the new features. The SVM math only needs to know how similar each pair of points is, and the kernel is a formula for that similarity. This keeps it fast, even when the bigger space would have LOTS of new features.

Common kernels in sklearn:

| Kernel | What it does |
|---------|-------------|
| `'linear'` | no new features, straight boundaries |
| `'poly'` | like adding polynomial features (eg. $x_1^2$, $x_1 x_2$, $x_2^2$), like we did with `PolynomialFeatures` |
| `'rbf'` (Gaussian) | smooth, curved boundaries, this is the default |

The RBF kernel measures how CLOSE two points are:

$$
K = e^{-\gamma \cdot (\text{distance between the two points})^2}
$$

If the two points are right on top of each other the distance is 0 and $K = e^0 = 1$. As the points get farther apart $K$ drops toward 0. So the RBF SVM is a little bit like KNN (Day 13): the prediction for a new point depends most on the support vectors that are close to it.

In [ ]:
# now try the RBF kernel
svm = SVC(kernel='rbf', gamma=1)
svm.fit(X_train, y_train)

print("Accuracy on the validation data:", accuracy_score(y_val, svm.predict(X_val)))
plot_boundary(svm, X_train, y_train, title="RBF SVM, gamma = 1")

Now we get a curved boundary that follows the ring, and the validation accuracy goes up to 0.85!

But where did gamma = 1 come from? I just picked it. So how do we know what is best?

### What does gamma do?

$\gamma$ (gamma) is in the RBF formula above. It controls how far away from a support vector a new point can be and still be affected by it.

- **Small gamma**: each support vector affects points far away, so the boundary is smooth and simple (risk of underfitting)
- **Large gamma**: each support vector only affects points very close to it, so the boundary can wiggle around single points (risk of overfitting)

Let's use a for loop to try some values.

In [ ]:
# What gamma should I use?
gammas = [0.1, 1, 10, 100, 1000]   # the values to try
train_acc = []                      # save the training accuracy here
val_acc = []                        # save the validation accuracy here
for g in gammas:
    svm = SVC(kernel='rbf', gamma=g)   # a new model for each gamma
    svm.fit(X_train, y_train)          # train on the TRAINING data
    train_acc.append(accuracy_score(y_train, svm.predict(X_train)))
    val_acc.append(accuracy_score(y_val, svm.predict(X_val)))

results = pd.DataFrame({'gamma': gammas, 'train accuracy': train_acc, 'validation accuracy': val_acc})
results

In [ ]:
plt.plot(gammas, train_acc, 'b-o', label='Training')
plt.plot(gammas, val_acc, 'r-o', label='Validation')
plt.xscale('log')
plt.xlabel("gamma")
plt.ylabel("Accuracy")
plt.legend()
plt.grid()
plt.show()

The training accuracy keeps going UP as gamma gets bigger, but the validation accuracy is best at $\gamma = 10$ and then goes back down. That should look familiar... it is overfitting! Let's look at what $\gamma = 1000$ is doing.

In [ ]:
svm = SVC(kernel='rbf', gamma=1000)
svm.fit(X_train, y_train)
plot_boundary(svm, X_train, y_train, title="RBF SVM, gamma = 1000")

Look closely: every red training point has its own tiny red circle around it, and everything else is blue. The model memorized the training data, so a new red point that lands between those circles gets called blue.

## You Try

Set `gamma=10` and try `C` = 0.01, 1, and 100 in the RBF SVM. Make the boundary plot for each one and check the validation accuracy. What does a very small C do here?

### The one test at the end

We pick $\gamma = 10$ because it had the best validation accuracy. NOW we get to use the test set, once.

In [ ]:
# Our final choice: gamma = 10. Train it and use the test set ONE time.
svm = SVC(kernel='rbf', gamma=10)
svm.fit(X_train, y_train)
print(f"Accuracy on the test set: {accuracy_score(y_test, svm.predict(X_test)):.3f}")

---------------------------------------

## SVMs NEED scaling

Remember Day 13: ALWAYS SCALE YOUR FEATURE DATA. For SVMs this really matters. The margin is a distance, and the RBF kernel is a distance, so a feature measured in the thousands will count much more than a feature measured in fractions.


---------------------------------------

## Where SVMs shine - Handwritten Digits

Let's try an SVM on a bigger example. sklearn has a data set of 1,797 handwritten digits (0 to 9). Each picture is 8 pixels by 8 pixels, so each one has **64 features**, one for how dark each pixel is. This is the kind of data where SVMs do really well: lots of features and not a huge number of rows.

We will compare an SVM to a decision tree from Day 15.

NOTE: There are 10 classes here, not 2! An SVM only draws a boundary between two classes, so behind the scenes sklearn trains one SVM for every pair of digits (0 vs 1, 0 vs 2, ...) and lets them vote. You use it exactly the same way.

In [ ]:
digits = load_digits()
X = digits.data      # 64 pixel values for each picture, already flattened (each 8x8 picture is one row of 64 numbers)
y = digits.target    # the digit, 0 to 9
print(X.shape)

# Let's look at the first 10 pictures
fig, axes = plt.subplots(1, 10, figsize=(12, 2))
for i in range(10):
    axes[i].imshow(digits.images[i], cmap='gray_r')   # the 8x8 picture
    axes[i].set_title(digits.target[i])               # the label
    axes[i].axis('off')
plt.show()

Same steps as always. Split the data 80/10/10 and scale the features (fit the scaler on the TRAINING data only).

NOTE: Remember from Day 15 that trees do not need scaling. It does not hurt them either, so we can give both models the same scaled data.

In [ ]:
X_train, X_test_full, y_train, y_test_full = train_test_split(
    X, y, test_size=0.2, random_state=random_seed
)
X_test, X_val, y_test, y_val = train_test_split(
    X_test_full, y_test_full, test_size=0.5, random_state=random_seed
)
print(f"Train: {len(X_train)}, Validation: {len(X_val)}, Test: {len(X_test)}")

# fit the scaler on the TRAINING data only, then transform everything
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

First the decision tree. What depth should we use?

In [ ]:
# What depth should I use?
depths = [5, 10, 15, 20]
tree_train = []
tree_val = []
for d in depths:
    tree = DecisionTreeClassifier(max_depth=d, random_state=42)   # a new tree for each depth
    tree.fit(X_train_s, y_train)
    tree_train.append(accuracy_score(y_train, tree.predict(X_train_s)))
    tree_val.append(accuracy_score(y_val, tree.predict(X_val_s)))

pd.DataFrame({'max_depth': depths, 'train accuracy': tree_train, 'validation accuracy': tree_val})

Now the SVM. We use the RBF kernel and try some values of C. (We leave gamma at the sklearn default, `gamma='scale'`, which picks a value based on the data.)

In [ ]:
# What C should I use?
Cs = [0.1, 1, 10, 100]
svm_train = []
svm_val = []
for c in Cs:
    svm = SVC(kernel='rbf', C=c)   # a new SVM for each C
    svm.fit(X_train_s, y_train)
    svm_train.append(accuracy_score(y_train, svm.predict(X_train_s)))
    svm_val.append(accuracy_score(y_val, svm.predict(X_val_s)))

pd.DataFrame({'C': Cs, 'train accuracy': svm_train, 'validation accuracy': svm_val})

WOW! The SVM gets more than 99% of the validation pictures right, and the best tree gets less than 80%. Notice that the deeper trees get 100% of the training data right but do not get any better on the validation data. That should look familiar... overfitting!

### The one test at the end

We pick the simplest model with the best validation accuracy for each one: a tree with `max_depth=10` and an SVM with `C=1` (remember, a smaller C is a simpler model). NOW we get to use the test set, once.

In [ ]:
# Our final choices. Train them and use the test set ONE time.
tree = DecisionTreeClassifier(max_depth=10, random_state=42)
tree.fit(X_train_s, y_train)
print(f"Decision tree - accuracy on the test set: {accuracy_score(y_test, tree.predict(X_test_s)):.3f}")

svm = SVC(kernel='rbf', C=1)
svm.fit(X_train_s, y_train)
print(f"SVM           - accuracy on the test set: {accuracy_score(y_test, svm.predict(X_test_s)):.3f}")

The SVM wins again. Notice that the tree did better on the test set than on the validation set. The validation and test sets only have 180 pictures each, so the numbers can jump around a lot depending on which pictures land in which set. This is why we will learn about cross-validation on Days 20 and 21!

NOTE - here the decision tree is getting a big boost from a lop sided test set - we will talk about cross-validation soon!

## You Try

Make a confusion matrix for the SVM on the test set with `confusion_matrix(y_test, svm.predict(X_test_s))`. Which digits does it mix up? Do those mistakes make sense when you think about how people write those numbers?

---------------------------------------

## Comparing Decision Trees and Support Vector Machines

Both Decision Trees and SVMs are powerful models for supervised learning, but they make their predictions in very different ways.

| | Decision Tree | Support Vector Machine |
|----------|----------------|------------------------|
| **How it works** | Splits the feature space into regions with yes or no questions (eg. "is feature 1 <= 3.2?") | Finds the boundary that maximizes the margin between classes |
| **Decision boundary** | Axis-aligned rectangles | Linear, or curved with a kernel |
| **Interpretability** | Very interpretable, you can see the rules | Harder to interpret |
| **Feature scaling** | Not needed | Required |
| **Main parameters** | `max_depth`, `ccp_alpha` | `C`, `kernel`, `gamma` |
| **Large data** | Fast | Slow when you have lots of rows |

In practice:
- For **tabular data** with lots of rows, trees (and the ensembles of trees you will learn about later) usually do better.
- For **smaller, well separated, or high dimensional data**, SVMs often do really well.

---------------------------------------

## Homework 16

The full assignment is in `HW_day16.ipynb` in your sandbox. You will use the breast cancer data (built into sklearn) to compare a linear SVM and an RBF SVM, choosing C and gamma with a validation set.

Work the problems in your sandbox and push it, your sandbox is your backup. Your write-up notebook goes in `Week09` of your team's repo, along with Homework 15. Both are due **Sunday 11/15 at 11:59pm**.